# Categorical robustness evaluation

This notebook evaluates the adapter checkpoints on a shared categorical test set and saves the outputs locally.

The categorical split keeps the underlying task fixed and applies four corruption types to the task text:

- typos,
- punctuation removal,
- shorthand substitutions,
- word drop.

For each adapter and condition, the notebook records JSON-format compliance and task-grounded content metrics. The main outputs are row-level predictions, summary tables with confidence intervals, a valid-JSON-only summary table, a delta table relative to the `mixed_0` baseline, and saved heatmaps.


## Local setup

Run this notebook from a local Jupyter environment. The commands below are meant to be run in a terminal before opening the notebook.

### 1. Create and activate a virtual environment

```bash
python -m venv .venv
source .venv/bin/activate
```

### 2. Install packages

```bash
pip install transformers datasets peft bitsandbytes accelerate bert-score evaluate rouge-score openai scikit-learn matplotlib pandas numpy jupyter ipykernel
```

### 3. Set environment variables

At minimum, set a project directory and a Hugging Face token for the base model. The OpenAI key is only needed when judge-based metrics are enabled.

```bash
export ROBUSTNESS_PROJECT_DIR="$HOME/llama_robustness_project"
export ADAPTER_ROOT="$ROBUSTNESS_PROJECT_DIR"
export HF_HOME="$ROBUSTNESS_PROJECT_DIR/.cache/huggingface"
export HF_TOKEN="your_huggingface_token"
export OPENAI_API_KEY="your_openai_key"   # optional if ENABLE_LLM_JUDGE=0
export MODEL_ID="meta-llama/Llama-3.2-3B-Instruct"
```

### 4. Launch Jupyter

```bash
jupyter notebook
```

### 5. Expected local files

This notebook looks for adapter folders under `ADAPTER_ROOT` and writes outputs under `ROBUSTNESS_PROJECT_DIR`.

Expected adapter paths:

- `llama3-3b-adapter-noise-0`
- `llama3-3b-adapter-noise-25`
- `llama3-3b-adapter-noise-50`
- `llama3-3b-adapter-noise-75`

Optional single-noise adapters, if present, are included automatically:

- `llama3-3b-adapter-typo-only`
- `llama3-3b-adapter-punctuation-only`
- `llama3-3b-adapter-shorthand-only`
- `llama3-3b-adapter-word-drop-only`

This notebook assumes an NVIDIA CUDA environment because the model is loaded in 4-bit mode.


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# -------------------------------
# Local paths and runtime settings
# -------------------------------
model_id = os.environ.get("MODEL_ID", "meta-llama/Llama-3.2-3B-Instruct")

project_dir = Path(
    os.environ.get("ROBUSTNESS_PROJECT_DIR", "./llama_robustness_project")
).expanduser().resolve()
adapter_root = Path(
    os.environ.get("ADAPTER_ROOT", str(project_dir))
).expanduser().resolve()
hf_home = Path(
    os.environ.get("HF_HOME", str(project_dir / ".cache" / "huggingface"))
).expanduser().resolve()
figures_dir = project_dir / "figures"

project_dir.mkdir(parents=True, exist_ok=True)
adapter_root.mkdir(parents=True, exist_ok=True)
hf_home.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_home)
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

strict_eval_path = project_dir / "strict_eval_set_tasknoise_answer_schema.csv"
categorical_eval_path = project_dir / "categorical_eval_set_taskgrounded_answer_schema.csv"
categorical_raw_path = project_dir / "categorical_noise_eval_taskgrounded_batch32.csv"
categorical_summary_path = project_dir / "categorical_noise_summary_with_ci_taskgrounded_batch32.csv"
categorical_conditioned_summary_path = project_dir / "categorical_noise_summary_valid_json_only_taskgrounded_batch32.csv"
categorical_delta_path = project_dir / "categorical_noise_delta_vs_mixed0_taskgrounded_batch32.csv"
categorical_validation_merge_path = project_dir / "categorical_noise_summary_with_validation_loss_taskgrounded_batch32.csv"
training_metadata_path = project_dir / "training_variants_metadata.csv"

# -------------------------------
# Reproducibility / runtime knobs
# -------------------------------
GLOBAL_SEED = 42
BOOTSTRAP_SAMPLES = 1000
STRICT_EVAL_SIZE = 200
EVAL_SAMPLE_SIZE = 200
INFERENCE_BATCH_SIZE = 32
MAX_INPUT_LENGTH = 768
MAX_NEW_TOKENS = 180
TEMPERATURE = 0.0

# Scoring controls
ENABLE_LLM_JUDGE = os.environ.get("ENABLE_LLM_JUDGE", "1") == "1"
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4o-mini")

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

print(f"Project directory: {project_dir}")
print(f"Adapter root: {adapter_root}")
print(f"Figures directory: {figures_dir}")
print(f"Batch size: {INFERENCE_BATCH_SIZE}")
print(f"Judge metrics enabled: {ENABLE_LLM_JUDGE}")


## Load the tokenizer and define the adapter-loading helper

Each adapter is evaluated on a fresh copy of the same base model so that the comparisons stay consistent.

The tokenizer uses left padding. That matters for batched generation with decoder-only models because the generated continuation needs to be sliced off after the shared padded prompt length.


In [ ]:
import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook expects an NVIDIA CUDA environment because the model is loaded in 4-bit mode."
    )

device = "cuda"
print(f"Using device: {device}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    token=os.environ.get("HF_TOKEN"),
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer padding_side: {tokenizer.padding_side}")
print(f"pad_token_id: {tokenizer.pad_token_id}, eos_token_id: {tokenizer.eos_token_id}")

def load_model_with_adapter(adapter_path: str):
    adapter_path = str(Path(adapter_path).expanduser().resolve())
    if not os.path.exists(adapter_path):
        raise FileNotFoundError(f"Adapter path does not exist: {adapter_path}")

    print(f"Loading base model with adapter: {adapter_path}")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model


## Define the evaluation metrics

The generated output is checked in two stages.

First, the notebook verifies that the response is valid JSON with the required keys. Second, it extracts the `answer` field and compares that content against the gold reference. This separation is useful because formatting failures and task failures are related but not identical.

Judge-based metrics are optional. Set `ENABLE_LLM_JUDGE=0` in the environment if you want to skip them.


In [ ]:
import json
import re
from typing import Any, Dict, Optional, Tuple

from evaluate import load
from openai import OpenAI

bertscore = load("bertscore")
rouge = load("rouge")

def load_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key and ENABLE_LLM_JUDGE:
        raise ValueError(
            "Set OPENAI_API_KEY in your local environment before running judge-based metrics, "
            "or disable them with ENABLE_LLM_JUDGE=0."
        )
    return OpenAI(api_key=api_key) if api_key else None

client = load_openai_client()

def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_json_object(raw_text: str) -> Tuple[Optional[dict], str]:
    if raw_text is None:
        return None, ""

    text = str(raw_text).strip()
    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0].strip()

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
    else:
        candidate = text

    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return parsed, candidate
        return None, candidate
    except Exception:
        return None, candidate

def get_json_metrics(output: str, required_keys=("answer", "summary", "components")) -> Dict[str, Any]:
    parsed, candidate = extract_json_object(output)
    if parsed is None:
        return {
            "json_score": 0.0,
            "valid_json": 0.0,
            "parsed_json": None,
            "missing_keys": list(required_keys),
        }

    missing_keys = [k for k in required_keys if k not in parsed]
    if len(missing_keys) == 0:
        json_score = 1.0
        valid_json = 1.0
    else:
        json_score = 0.5
        valid_json = 0.0

    return {
        "json_score": json_score,
        "valid_json": valid_json,
        "parsed_json": parsed,
        "missing_keys": missing_keys,
    }

def extract_answer_text(output: str) -> str:
    json_metrics = get_json_metrics(output)
    parsed = json_metrics["parsed_json"]
    if parsed is None:
        return normalize_text(output)

    answer = parsed.get("answer", "")
    summary = parsed.get("summary", "")
    components = parsed.get("components", [])

    pieces = []
    if isinstance(answer, str) and normalize_text(answer):
        pieces.append(normalize_text(answer))
    if isinstance(summary, str) and normalize_text(summary) and normalize_text(summary) not in pieces:
        pieces.append(normalize_text(summary))
    if isinstance(components, list):
        component_text = "; ".join([normalize_text(x) for x in components if normalize_text(x)])
        if component_text:
            pieces.append(component_text)

    fallback = " ".join([p for p in pieces if p]).strip()
    return fallback if fallback else normalize_text(output)

def get_semantic_score(answer_text: str, reference: str) -> float:
    answer_text = normalize_text(answer_text)
    reference = normalize_text(reference)
    if not answer_text or not reference:
        return 0.0
    results = bertscore.compute(predictions=[answer_text], references=[reference], lang="en")
    return float(results["f1"][0])

def get_rouge_l_score(answer_text: str, reference: str) -> float:
    answer_text = normalize_text(answer_text)
    reference = normalize_text(reference)
    if not answer_text or not reference:
        return 0.0
    results = rouge.compute(predictions=[answer_text], references=[reference], use_aggregator=False)
    return float(results["rougeL"][0])

def get_dual_judge_scores(prompt: str, answer_text: str, reference: str) -> Dict[str, float]:
    if not ENABLE_LLM_JUDGE or client is None:
        return {"judge_score": np.nan, "reference_judge_score": np.nan}

    system_instruction = """
    You are an expert evaluator.
    Score the candidate answer on two axes from 1.0 to 5.0:
    1. prompt_quality: how well the answer addresses the user prompt.
    2. reference_alignment: how well the answer matches the gold reference answer.

    Output only a valid JSON object with exactly two keys:
    {"prompt_quality": <float>, "reference_alignment": <float>}
    """

    judge_prompt = f"""
    <user_prompt>
    {prompt}
    </user_prompt>

    <candidate_answer>
    {answer_text}
    </candidate_answer>

    <gold_reference_answer>
    {reference}
    </gold_reference_answer>
    """

    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": judge_prompt},
            ],
            temperature=0.0,
        )
        content = response.choices[0].message.content.strip()
        parsed, _ = extract_json_object(content)
        if parsed is None:
            raise ValueError(f"Could not parse judge JSON from: {content}")

        pq = float(parsed.get("prompt_quality", 1.0))
        ra = float(parsed.get("reference_alignment", 1.0))
        pq = max(1.0, min(5.0, pq))
        ra = max(1.0, min(5.0, ra))
        return {
            "judge_score": (pq - 1.0) / 4.0,
            "reference_judge_score": (ra - 1.0) / 4.0,
        }
    except Exception as e:
        print(f"Judge error: {e}")
        return {"judge_score": 0.0, "reference_judge_score": 0.0}

def score_example(judge_prompt: str, output: str, reference: str) -> Dict[str, Any]:
    json_metrics = get_json_metrics(output)
    answer_text = extract_answer_text(output)

    semantic_score = get_semantic_score(answer_text, reference)
    rougeL_score = get_rouge_l_score(answer_text, reference)
    judge_scores = get_dual_judge_scores(judge_prompt, answer_text, reference)

    valid_json = json_metrics["valid_json"]
    conditioned = {
        "semantic_score_valid_json": semantic_score if valid_json == 1.0 else np.nan,
        "judge_score_valid_json": judge_scores["judge_score"] if valid_json == 1.0 else np.nan,
        "reference_judge_score_valid_json": judge_scores["reference_judge_score"] if valid_json == 1.0 else np.nan,
        "rougeL_valid_json": rougeL_score if valid_json == 1.0 else np.nan,
    }

    return {
        "json_score": round(float(json_metrics["json_score"]), 4),
        "valid_json": round(float(valid_json), 4),
        "semantic_score": round(float(semantic_score), 4),
        "judge_score": round(float(judge_scores["judge_score"]), 4) if not np.isnan(judge_scores["judge_score"]) else np.nan,
        "reference_judge_score": round(float(judge_scores["reference_judge_score"]), 4) if not np.isnan(judge_scores["reference_judge_score"]) else np.nan,
        "rougeL": round(float(rougeL_score), 4),
        "extracted_answer": answer_text,
        "missing_keys": ", ".join(json_metrics["missing_keys"]),
        **conditioned,
    }

def bootstrap_mean_ci(values, n_boot=BOOTSTRAP_SAMPLES, ci=95, seed=GLOBAL_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(sample.mean())

    alpha = (100 - ci) / 2
    return (
        float(values.mean()),
        float(np.percentile(boot_means, alpha)),
        float(np.percentile(boot_means, 100 - alpha)),
    )

def summarize_with_ci(eval_df: pd.DataFrame, group_cols, metrics):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    rows = []
    for group_values, group_df in eval_df.groupby(group_cols, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        row = {col: val for col, val in zip(group_cols, group_values)}
        row["n_examples"] = int(len(group_df))
        for metric in metrics:
            mean_val, ci_low, ci_high = bootstrap_mean_ci(group_df[metric].values)
            row[f"{metric}_mean"] = round(mean_val, 4) if not np.isnan(mean_val) else np.nan
            row[f"{metric}_ci_low"] = round(ci_low, 4) if not np.isnan(ci_low) else np.nan
            row[f"{metric}_ci_high"] = round(ci_high, 4) if not np.isnan(ci_high) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

MAIN_METRICS = ["valid_json", "json_score", "semantic_score", "judge_score", "reference_judge_score", "rougeL"]
VALID_JSON_ONLY_METRICS = ["semantic_score_valid_json", "judge_score_valid_json", "reference_judge_score_valid_json", "rougeL_valid_json"]

print("Evaluation metrics loaded.")


## Build or reload the categorical evaluation set

This section creates a shared 200-example evaluation split and then derives four corruption families from the same underlying tasks.

The key design choice is that corruption is applied only to the task text. The answer-schema instruction is appended afterward and kept identical across conditions. That makes the comparison cleaner because differences across categories are driven by the task corruption rather than by changes to the JSON requirement.

If a saved evaluation file already exists locally and has the expected columns, the notebook reuses it. Otherwise it rebuilds the split and writes it to disk.


In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

JSON_CONSTRAINT = (
    "Respond ONLY as a valid JSON object with exactly three keys: "
    "'answer' (the full answer to the task as a string), "
    "'summary' (a brief summary string), and "
    "'components' (a list of key concepts)."
)

EXPECTED_SHARED_COLUMNS = {
    "instruction", "input", "output",
    "clean_prompt", "clean_judge_prompt",
}

EXPECTED_CATEGORICAL_COLUMNS = {
    "instruction", "input", "output",
    "clean_prompt", "clean_judge_prompt",
    "typos_prompt", "typos_judge_prompt",
    "punctuation_prompt", "punctuation_judge_prompt",
    "shorthand_prompt", "shorthand_judge_prompt",
    "word_drop_prompt", "word_drop_judge_prompt",
}

def normalize_text_for_prompt(text):
    if pd.isna(text):
        return ""
    return str(text).strip()

def typo_noise(text, rng):
    text = normalize_text_for_prompt(text)
    if len(text) < 4:
        return text
    chars = list(text)
    candidate_positions = [i for i in range(len(chars) - 1) if chars[i].isalpha() and chars[i + 1].isalpha()]
    if not candidate_positions:
        return text
    pos = int(rng.choice(candidate_positions))
    chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
    return "".join(chars)

def punctuation_noise(text, rng):
    text = normalize_text_for_prompt(text)
    return re.sub(r"[.,!?;:]", "", text)

def shorthand_noise(text, rng):
    text = normalize_text_for_prompt(text)
    replacements = {
        "you": "u",
        "are": "r",
        "please": "pls",
        "people": "ppl",
        "with": "w/",
        "without": "w/o",
        "before": "b4",
    }
    words = text.split()
    return " ".join([replacements.get(w.lower(), w) for w in words])

def word_drop_noise(text, rng):
    text = normalize_text_for_prompt(text)
    words = text.split()
    if len(words) <= 3:
        return text
    drop_idx = int(rng.integers(0, len(words)))
    words.pop(drop_idx)
    return " ".join(words)

def build_task_prompt(instruction, input_text):
    instruction = normalize_text_for_prompt(instruction)
    input_text = normalize_text_for_prompt(input_text)
    if input_text:
        return f"Instruction: {instruction}\nInput: {input_text}"
    return f"Instruction: {instruction}"

def build_inference_prompt(task_text):
    return f"{task_text}\n\n{JSON_CONSTRAINT}"

def build_judge_prompt(instruction, input_text):
    instruction = normalize_text_for_prompt(instruction)
    input_text = normalize_text_for_prompt(input_text)
    if input_text:
        return f"Instruction: {instruction}\nInput: {input_text}"
    return f"Instruction: {instruction}"

def build_categorical_prompts(base_df):
    eval_df = base_df[["instruction", "input", "output"]].copy()

    eval_df["clean_judge_prompt"] = eval_df.apply(
        lambda row: build_judge_prompt(row["instruction"], row["input"]), axis=1
    )
    eval_df["clean_prompt"] = eval_df.apply(
        lambda row: build_inference_prompt(build_task_prompt(row["instruction"], row["input"])), axis=1
    )

    condition_specs = {
        "typos": typo_noise,
        "punctuation": punctuation_noise,
        "shorthand": shorthand_noise,
        "word_drop": word_drop_noise,
    }

    seed_offsets = {
        "typos": 1000,
        "punctuation": 2000,
        "shorthand": 3000,
        "word_drop": 4000,
    }

    for condition, fn in condition_specs.items():
        noisy_instructions = []
        noisy_inputs = []
        for idx, row in eval_df.iterrows():
            rng = np.random.default_rng(GLOBAL_SEED + seed_offsets[condition] + idx)
            noisy_instruction = fn(row["instruction"], rng)
            noisy_input = fn(row["input"], rng) if normalize_text_for_prompt(row["input"]) else ""
            noisy_instructions.append(noisy_instruction)
            noisy_inputs.append(noisy_input)

        eval_df[f"{condition}_instruction"] = noisy_instructions
        eval_df[f"{condition}_input"] = noisy_inputs
        eval_df[f"{condition}_judge_prompt"] = eval_df.apply(
            lambda row, c=condition: build_judge_prompt(row[f"{c}_instruction"], row[f"{c}_input"]),
            axis=1,
        )
        eval_df[f"{condition}_prompt"] = eval_df.apply(
            lambda row, c=condition: build_inference_prompt(
                build_task_prompt(row[f"{c}_instruction"], row[f"{c}_input"])
            ),
            axis=1,
        )

    return eval_df

categorical_df = None

if categorical_eval_path.exists():
    cached_df = pd.read_csv(categorical_eval_path)
    if EXPECTED_CATEGORICAL_COLUMNS.issubset(set(cached_df.columns)) and len(cached_df) == STRICT_EVAL_SIZE:
        categorical_df = cached_df.copy()
        print(f"Loaded existing categorical evaluation set from: {categorical_eval_path}")

if categorical_df is None:
    shared_df = None
    if strict_eval_path.exists():
        candidate_df = pd.read_csv(strict_eval_path)
        if EXPECTED_SHARED_COLUMNS.issubset(set(candidate_df.columns)) and len(candidate_df) == STRICT_EVAL_SIZE:
            shared_df = candidate_df[["instruction", "input", "output"]].copy()
            print(f"Loaded shared strict evaluation set from: {strict_eval_path}")

    if shared_df is None:
        print("Shared strict evaluation set not found. Rebuilding the 200-example holdout split.")
        dataset = load_dataset("yahma/alpaca-cleaned", split="train")
        df = dataset.to_pandas()[["instruction", "input", "output"]].copy()
        df["instruction"] = df["instruction"].fillna("").astype(str).str.strip()
        df["input"] = df["input"].fillna("").astype(str).str.strip()
        df["output"] = df["output"].fillna("").astype(str).str.strip()

        train_df, holdout_df = train_test_split(df, test_size=0.10, random_state=GLOBAL_SEED)
        _, shared_df = train_test_split(holdout_df, test_size=STRICT_EVAL_SIZE, random_state=GLOBAL_SEED)
        shared_df = shared_df.reset_index(drop=True).copy()

    categorical_df = build_categorical_prompts(shared_df.reset_index(drop=True))
    categorical_df.to_csv(categorical_eval_path, index=False)
    print(f"Saved categorical evaluation set to: {categorical_eval_path}")

if EVAL_SAMPLE_SIZE < len(categorical_df):
    categorical_df = categorical_df.sample(n=EVAL_SAMPLE_SIZE, random_state=GLOBAL_SEED).reset_index(drop=True)
    print(f"Using a reproducible subset of {EVAL_SAMPLE_SIZE} examples for evaluation.")
else:
    categorical_df = categorical_df.reset_index(drop=True).copy()

print(f"Final categorical evaluation size: {len(categorical_df)} examples")
display(
    categorical_df[
        [
            "instruction", "input",
            "typos_instruction", "punctuation_instruction",
            "shorthand_instruction", "word_drop_instruction",
        ]
    ].head(3)
)


## Sanity-check the categorical prompts

Before running the model, it helps to inspect a few examples and one full prompt per condition.

This check confirms two things:
1. the corruption is applied only to the task text, and
2. the JSON constraint is still present in every condition.


In [ ]:
preview_cols = [
    "instruction", "input",
    "typos_instruction", "punctuation_instruction",
    "shorthand_instruction", "word_drop_instruction",
]
display(categorical_df[preview_cols].sample(n=5, random_state=GLOBAL_SEED))

sample_idx = 0
for cond in ["clean", "typos", "punctuation", "shorthand", "word_drop"]:
    print(f"\n----- {cond.upper()} INFERENCE PROMPT -----")
    print(categorical_df.loc[sample_idx, f"{cond}_prompt"][:1000])

print(
    "\nJSON constraint present in every condition:",
    all(
        JSON_CONSTRAINT in categorical_df.loc[sample_idx, f"{cond}_prompt"]
        for cond in ["clean", "typos", "punctuation", "shorthand", "word_drop"]
    ),
)


## Run batched categorical evaluation across adapters

This is the main evaluation loop. For each adapter, the notebook generates outputs for all five conditions, scores them, and appends the results to the local summary tables.

Generation starts at batch size 32. If a CUDA out-of-memory error occurs, the loop reduces the batch size and retries so the run can continue without manual edits.


In [ ]:
from tqdm.auto import tqdm

adapter_specs = [
    {"label": "mixed_0", "path": adapter_root / "llama3-3b-adapter-noise-0", "training_noise_ratio": 0},
    {"label": "mixed_25", "path": adapter_root / "llama3-3b-adapter-noise-25", "training_noise_ratio": 25},
    {"label": "mixed_50", "path": adapter_root / "llama3-3b-adapter-noise-50", "training_noise_ratio": 50},
    {"label": "mixed_75", "path": adapter_root / "llama3-3b-adapter-noise-75", "training_noise_ratio": 75},
]

optional_single_noise_adapters = [
    {"label": "typo_only", "path": adapter_root / "llama3-3b-adapter-typo-only", "training_noise_ratio": np.nan},
    {"label": "punctuation_only", "path": adapter_root / "llama3-3b-adapter-punctuation-only", "training_noise_ratio": np.nan},
    {"label": "shorthand_only", "path": adapter_root / "llama3-3b-adapter-shorthand-only", "training_noise_ratio": np.nan},
    {"label": "word_drop_only", "path": adapter_root / "llama3-3b-adapter-word-drop-only", "training_noise_ratio": np.nan},
]

for spec in optional_single_noise_adapters:
    if spec["path"].exists():
        adapter_specs.append(spec)
        print(f"Found optional adapter: {spec['label']}")

condition_specs = [
    ("clean", "clean_prompt", "clean_judge_prompt"),
    ("typos", "typos_prompt", "typos_judge_prompt"),
    ("punctuation", "punctuation_prompt", "punctuation_judge_prompt"),
    ("shorthand", "shorthand_prompt", "shorthand_judge_prompt"),
    ("word_drop", "word_drop_prompt", "word_drop_judge_prompt"),
]

def build_inference_jobs(eval_df):
    jobs = []
    for index, row in eval_df.iterrows():
        for condition, prompt_col, judge_col in condition_specs:
            jobs.append(
                {
                    "id": index,
                    "condition": condition,
                    "prompt": row[prompt_col],
                    "judge_prompt": row[judge_col],
                    "target_output": row["output"],
                }
            )
    return pd.DataFrame(jobs)

def generate_in_batches(model, prompts, batch_size=INFERENCE_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    all_text = []
    idx = 0
    current_batch_size = batch_size

    pbar = tqdm(total=len(prompts), desc="Batched generation")
    while idx < len(prompts):
        take = min(current_batch_size, len(prompts) - idx)
        batch_prompts = prompts[idx: idx + take]

        try:
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            generation_kwargs = dict(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            if TEMPERATURE and TEMPERATURE > 0:
                generation_kwargs["do_sample"] = True
                generation_kwargs["temperature"] = TEMPERATURE
                generation_kwargs["top_p"] = 0.95
            else:
                generation_kwargs["do_sample"] = False

            with torch.inference_mode():
                outputs = model.generate(**generation_kwargs)

            input_seq_len = inputs["input_ids"].shape[1]
            for row_idx in range(outputs.shape[0]):
                generated_ids = outputs[row_idx][input_seq_len:]
                text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                all_text.append(text)

            idx += take
            pbar.update(take)
            current_batch_size = batch_size

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if current_batch_size == 1:
                raise RuntimeError(
                    "Out of memory even at batch size 1. Reduce MAX_INPUT_LENGTH or MAX_NEW_TOKENS."
                )
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA out of memory. Retrying with batch size {current_batch_size}.")

    pbar.close()
    return all_text

jobs_df = build_inference_jobs(categorical_df)
print(f"Prepared {len(jobs_df)} inference jobs ({len(categorical_df)} examples x {len(condition_specs)} conditions).")

all_summaries = []
all_conditioned_summaries = []
master_eval_df = pd.DataFrame()

for spec in adapter_specs:
    print(f"\n{'=' * 60}")
    print(f"Evaluating adapter: {spec['label']}")
    print(f"{'=' * 60}")

    model = load_model_with_adapter(spec["path"])

    generated_outputs = generate_in_batches(
        model=model,
        prompts=jobs_df["prompt"].tolist(),
        batch_size=INFERENCE_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    adapter_results_df = jobs_df.copy()
    adapter_results_df["training_condition"] = spec["label"]
    adapter_results_df["training_noise_ratio"] = spec["training_noise_ratio"]
    adapter_results_df["generated_output"] = generated_outputs

    scored_rows = []
    for _, row in tqdm(
        adapter_results_df.iterrows(),
        total=len(adapter_results_df),
        desc=f"Scoring ({spec['label']})",
    ):
        metrics = score_example(
            judge_prompt=row["judge_prompt"],
            output=row["generated_output"],
            reference=row["target_output"],
        )
        row_data = row.to_dict()
        row_data.update(metrics)
        scored_rows.append(row_data)

    eval_df = pd.DataFrame(scored_rows)
    master_eval_df = pd.concat([master_eval_df, eval_df], ignore_index=True)

    summary = summarize_with_ci(eval_df, ["training_condition", "training_noise_ratio", "condition"], MAIN_METRICS)
    all_summaries.append(summary)

    conditioned_summary = summarize_with_ci(
        eval_df,
        ["training_condition", "training_noise_ratio", "condition"],
        VALID_JSON_ONLY_METRICS,
    )
    all_conditioned_summaries.append(conditioned_summary)

    print(f"Main summary for {spec['label']}:")
    display(summary)
    print(f"Content metrics conditioned on valid JSON for {spec['label']}:")
    display(conditioned_summary)

    del model
    gc.collect()
    torch.cuda.empty_cache()

final_master_summary = pd.concat(all_summaries, ignore_index=True)
final_master_summary = final_master_summary.sort_values(["training_condition", "condition"]).reset_index(drop=True)

final_conditioned_summary = pd.concat(all_conditioned_summaries, ignore_index=True)
final_conditioned_summary = final_conditioned_summary.sort_values(["training_condition", "condition"]).reset_index(drop=True)

master_eval_df.to_csv(categorical_raw_path, index=False)
final_master_summary.to_csv(categorical_summary_path, index=False)
final_conditioned_summary.to_csv(categorical_conditioned_summary_path, index=False)

print(f"Saved raw evaluations to: {categorical_raw_path}")
print(f"Saved main summary with confidence intervals to: {categorical_summary_path}")
print(f"Saved valid-JSON-only summary to: {categorical_conditioned_summary_path}")
print("Categorical evaluation complete.")


## Merge the categorical summary with clean validation loss

This step is optional. It is mainly useful for the mixed-noise adapters because it lets you compare category-level robustness against the clean validation loss selected during training.

If the training metadata file is present locally, the notebook tries to infer the noise-ratio column and the validation-loss column automatically.


In [ ]:
summary_with_validation = final_master_summary.copy()

if training_metadata_path.exists():
    training_meta = pd.read_csv(training_metadata_path)
    print("Training metadata preview:")
    display(training_meta.head())

    ratio_col = None
    loss_col = None
    for c in training_meta.columns:
        if c.lower() in {"noise_ratio", "training_noise_ratio", "ratio"}:
            ratio_col = c
        if c.lower() in {"best_eval_loss", "best_validation_loss", "eval_loss", "validation_loss"}:
            loss_col = c

    if ratio_col is not None and loss_col is not None:
        merged_loss = training_meta[[ratio_col, loss_col]].drop_duplicates().rename(
            columns={ratio_col: "training_noise_ratio", loss_col: "best_validation_loss"}
        )
        summary_with_validation = summary_with_validation.merge(
            merged_loss,
            on="training_noise_ratio",
            how="left",
        )
        summary_with_validation.to_csv(categorical_validation_merge_path, index=False)
        print(f"Saved summary merged with validation loss to: {categorical_validation_merge_path}")
        display(summary_with_validation)
    else:
        print("Could not infer noise-ratio and validation-loss columns from training metadata.")
        print("Available columns:", training_meta.columns.tolist())
else:
    print(f"Training metadata file not found at: {training_metadata_path}")


## Compute deltas versus the `mixed_0` baseline

This table gives a direct comparison against the no-noise training baseline. For each corruption condition, it subtracts the `mixed_0` score from every other adapter on the same metric.

Positive deltas mean the adapter outperformed the baseline on that condition. Negative deltas mean it underperformed the baseline.


In [ ]:
key_metrics_for_delta = [
    "valid_json_mean",
    "semantic_score_mean",
    "judge_score_mean",
    "reference_judge_score_mean",
    "rougeL_mean",
]

baseline = final_master_summary[final_master_summary["training_condition"] == "mixed_0"].copy()
baseline = baseline[["condition"] + key_metrics_for_delta].rename(
    columns={m: f"baseline_{m}" for m in key_metrics_for_delta}
)

delta_df = final_master_summary.merge(baseline, on="condition", how="left")
for metric in key_metrics_for_delta:
    delta_df[f"delta_{metric}"] = delta_df[metric] - delta_df[f"baseline_{metric}"]

delta_cols = [
    "training_condition", "training_noise_ratio", "condition",
    *key_metrics_for_delta,
    *[f"delta_{m}" for m in key_metrics_for_delta],
]
delta_df = delta_df[delta_cols].sort_values(["training_condition", "condition"]).reset_index(drop=True)
delta_df.to_csv(categorical_delta_path, index=False)

print(f"Saved categorical delta table to: {categorical_delta_path}")
display(delta_df)


## Visualize categorical tradeoffs and save the figures

The heatmaps below show the results from three angles:

- `valid_json_mean` measures schema adherence,
- `reference_judge_score_mean` measures end-to-end task quality, and
- `reference_judge_score_valid_json_mean` measures task quality on the subset of responses that stayed in schema.

Each figure is displayed in the notebook and also saved to the local `figures/` directory.


In [ ]:
import matplotlib.pyplot as plt

def plot_heatmap(summary_df, value_col, title, filename, order=None):
    heat_df = summary_df.pivot(index="training_condition", columns="condition", values=value_col)
    if order is not None:
        ordered_cols = [c for c in order if c in heat_df.columns]
        heat_df = heat_df[ordered_cols]

    fig, ax = plt.subplots(figsize=(9, 4 + 0.5 * len(heat_df)))
    image = ax.imshow(heat_df.values, aspect="auto")
    fig.colorbar(image, ax=ax, label=value_col)
    ax.set_xticks(range(len(heat_df.columns)))
    ax.set_xticklabels(heat_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heat_df.index)))
    ax.set_yticklabels(heat_df.index)
    ax.set_title(title)
    fig.tight_layout()

    out_path = figures_dir / filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {out_path}")
    plt.show()

condition_order = ["clean", "typos", "punctuation", "shorthand", "word_drop"]

plot_heatmap(
    final_master_summary,
    "valid_json_mean",
    "Categorical valid JSON rate",
    "categorical_valid_json_rate.png",
    order=condition_order,
)

plot_heatmap(
    final_master_summary,
    "reference_judge_score_mean",
    "Categorical reference-grounded judge score",
    "categorical_reference_grounded_judge_score.png",
    order=condition_order,
)

plot_heatmap(
    final_conditioned_summary,
    "reference_judge_score_valid_json_mean",
    "Reference-grounded judge score conditioned on valid JSON",
    "categorical_reference_grounded_judge_score_valid_json_only.png",
    order=condition_order,
)


## Review the final categorical tables

The first table gives the unconditional end-to-end view across categories. The second keeps only the subset of responses that were valid JSON.

If the optional validation-loss merge was available, the merged table can be inspected afterward to compare robustness patterns against training-time clean validation loss.


In [ ]:
print("Main category-level summary:")
display(final_master_summary)

print("\nContent metrics conditioned on valid JSON:")
display(final_conditioned_summary)

if "summary_with_validation" in globals():
    print("\nSummary merged with clean validation loss (if available):")
    display(summary_with_validation)
